# Desafío · Una fuente cambió sin avisar

La aplicación publicó una versión nueva. El lote contiene un tipo de evento nuevo (`refund`), una columna adicional (`app_version`) y al menos un evento duplicado. Incorporalo sin perder la historia ni duplicar identificadores.

In [0]:
dbutils.widgets.text("student_id", "alumno01", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
challenge_path = f"{config.volume_path}/events_json_v2"
build_challenge_batch(spark, config).coalesce(2).write.mode("overwrite").json(challenge_path)
print(challenge_path)

/Volumes/workspace/bigdata_drrach/landing/events_json_v2


## Consigna 1 — Contrato y diagnóstico

Leé el lote nuevo, compará su esquema con `bronze_events` y respondé:

1. ¿Qué campo apareció?
2. ¿Qué valor de `event_type` no existía?
3. ¿Cuántas filas e identificadores distintos contiene?

In [0]:
# TODO: leer challenge_path, inspeccionar el esquema y calcular las tres respuestas.
challenge = None

## Consigna 2 — Integración controlada

Construí `bronze_events_v2` con estas propiedades:

- Contiene los eventos históricos y el lote nuevo.
- Conserva `app_version`, usando `NULL` para la historia anterior.
- Tiene un único registro por `event_id`.
- Agrega `_source`, `_source_file` y `_ingested_at`.
- Puede volver a ejecutarse sin cambiar el total final.

No modifiques `bronze_events`: queremos conservarlo como evidencia de la primera ingesta.

In [0]:
from pyspark.sql import functions as F

source_table = f"{config.catalog}.{config.schema}.bronze_events"
target_table = f"{config.catalog}.{config.schema}.bronze_events_v2"

# TODO: unir por nombre admitiendo columnas faltantes, deduplicar y escribir target_table.

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6542377359554649>, line 3
      1 from pyspark.sql import functions as F
----> 3 source_table = f"{config.catalog}.{config.schema}.bronze_events"
      4 target_table = f"{config.catalog}.{config.schema}.bronze_events_v2"

NameError: name 'config' is not defined

## Consigna 3 — Verificación automática

Completá las aserciones. Una entrega válida debe demostrar que no hay duplicados y que la columna evolucionada existe.

In [0]:
# TODO: reemplazar los valores pendientes.
result = spark.table(target_table)
total_rows = None
distinct_ids = None
assert total_rows == distinct_ids, "Hay event_id duplicados"
assert "app_version" in result.columns, "No se preservó la evolución del esquema"
assert result.where("event_type = 'refund'").count() > 0, "No se incorporó refund"

## Reflexión final

En no más de 150 palabras, explicá la diferencia entre: detectar una evolución de esquema, aceptarla técnicamente y decidir que es válida para el negocio.

In [0]:
from pyspark.sql import functions as F

tx = spark.table("workspace.bigdata_drrach.bronze_transactions")

resultado = tx.agg(
    F.count("*").alias("rows"),
    F.countDistinct("transaction_id").alias("distinct_ids"),
    F.sum(
        F.expr("CASE WHEN try_cast(amount AS DECIMAL(12,2)) IS NULL THEN 1 ELSE 0 END")
    ).alias("invalid_amounts")
).first()

print("Filas:", resultado["rows"])
print("IDs distintos:", resultado["distinct_ids"])
print("Importes inválidos:", resultado["invalid_amounts"])

Filas: 1001
IDs distintos: 1000
Importes inválidos: 3


In [0]:
result = spark.table("workspace.bigdata_drrach.bronze_events_v2")

total = result.count()
distintos = result.select("event_id").distinct().count()
refunds = result.filter("event_type = 'refund'").count()

print("Eventos totales:", total)
print("IDs distintos:", distintos)
print("Eventos refund:", refunds)
print("Sin duplicados:", total == distintos)

Eventos totales: 3150
IDs distintos: 3150
Eventos refund: 38
Sin duplicados: True


In [0]:
print("Total:", result.count())
print("IDs distintos:", result.select("event_id").distinct().count())
print("Refund:", result.filter("event_type = 'refund'").count())

Total: 200000
IDs distintos: 200000
Refund: 38


In [0]:
historia = spark.table("workspace.bigdata_drrach.bronze_events")
nuevo = spark.table("workspace.bigdata_drrach.bronze_events_v2")

faltantes = historia.select("event_id").distinct().subtract(
    nuevo.select("event_id").distinct()
).count()

print("Eventos históricos:", historia.count())
print("Históricos faltantes en v2:", faltantes)
print("Eventos v2:", nuevo.count())
print("App_version informado:", nuevo.filter("app_version IS NOT NULL").count())

Eventos históricos: 3000
Históricos faltantes en v2: 0
Eventos v2: 3150
App_version informado: 150


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

base = "workspace.bigdata_drrach"
ruta_nueva = "/Volumes/workspace/bigdata_drrach/landing/events_json_v2"

historia = spark.table(f"{base}.bronze_events")
lote = spark.read.json(ruta_nueva)

print("Historia:", historia.count())
print("Filas del lote nuevo:", lote.count())
print("IDs distintos del lote:", lote.select("event_id").distinct().count())

lote_con_metadatos = (
    lote
    .withColumn("_source", F.lit("events_json_v2"))
    .withColumn("_source_file", F.lit(ruta_nueva))
    .withColumn("_ingested_at", F.current_timestamp())
)

unidos = (
    historia.withColumn("_prioridad", F.lit(0))
    .unionByName(
        lote_con_metadatos.withColumn("_prioridad", F.lit(1)),
        allowMissingColumns=True
    )
)

ventana = Window.partitionBy("event_id").orderBy(F.desc("_prioridad"))

final = (
    unidos
    .withColumn("_fila", F.row_number().over(ventana))
    .filter(F.col("_fila") == 1)
    .drop("_fila", "_prioridad")
)

(
    final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{base}.bronze_events_v2")
)

verificacion = spark.table(f"{base}.bronze_events_v2")
print("Total v2:", verificacion.count())
print("IDs distintos v2:", verificacion.select("event_id").distinct().count())
print("Refund:", verificacion.filter(F.col("event_type") == "refund").count())

Historia: 3000
Filas del lote nuevo: 151
IDs distintos del lote: 150
Total v2: 3150
IDs distintos v2: 3150
Refund: 38


In [0]:
result = spark.table("workspace.bigdata_drrach.bronze_events_v2")

assert result.count() == result.select("event_id").distinct().count(), "Hay event_id duplicados"
assert "app_version" in result.columns, "Falta app_version"
assert result.filter("event_type = 'refund'").count() > 0, "Falta refund"

print("Verificación superada")

Verificación superada
